# FORESIGHT — AI-Powered Demand & Inventory Intelligence Platform

**End-to-end Jupyter Notebook | Python | pandas | LightGBM | TensorFlow/Keras LSTM | Streamlit**

This notebook implements the FORESIGHT workflow against the supplied synthetic retail dataset. The project brief defines FORESIGHT as a four-week client engagement for NorthBay Living, with a weekly SKU-level demand forecast, stockout/overstock risk scoring, a planning dashboard, and a deployable scoring layer. fileciteturn0file0L69-L83

> **Important dataset decision:** the supplied project brief explicitly says the synthetic dataset is provided and that generating data is not the engagement task. Therefore, the main pipeline loads the provided CSV extracts. A compact synthetic-data fallback generator is included only so the notebook can be demonstrated independently. fileciteturn0file0L196-L213

The supplied dataset contains store, SKU, customer, promotion, inventory and transaction extracts, including a ground-truth inventory anomaly file. fileciteturn0file1L12-L35


## Business Objective

FORESIGHT converts historical sales and inventory data into:

1. Weekly SKU-level demand forecasts.
2. Stockout-risk and overstock-risk signals.
3. Reorder Point (ROP) and Economic Order Quantity (EOQ) recommendations.
4. Prioritised replenishment / clearance actions.
5. A Streamlit operations dashboard.

The project brief requires honest backtesting against a seasonal-naive baseline and prohibits future-data leakage. fileciteturn0file0L228-L254


In [ ]:
# ============================================================
# STEP 0 — Environment Setup
# ============================================================
# If TensorFlow is not installed, run this once in a terminal:
# pip install -r requirements.txt

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)

try:
    import lightgbm as lgb
    print("LightGBM:", lgb.__version__)
except ImportError:
    print("LightGBM is not installed. Run: pip install lightgbm")

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except ImportError:
    print("TensorFlow is not installed yet. The LSTM cells will install/import it when available.")

sns.set_theme(style="whitegrid", context="notebook")


## STEP 1 — Load the supplied synthetic dataset

The supplied dataset is relational: transactions link to stores, SKUs, customers and promotions, while inventory links to store/SKU pairs. The README describes 5,000 SKUs, 30 stores, 10,000 customers and the transaction/inventory schemas. fileciteturn0file1L16-L35 fileciteturn0file1L118-L132

The code below searches the project `data/` directory first and then `/mnt/data/`, so it works both in this delivered project and in the current Jupyter environment.


In [ ]:
# ============================================================
# STEP 1A — File discovery and loading
# ============================================================
def locate_file(filename):
    candidates = [
        DATA_DIR / filename,
        Path("/mnt/data") / filename,
        Path.cwd() / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

FILES = {
    "sales": "sales_transactions.csv",
    "sku": "sku_master.csv",
    "store": "store_master.csv",
    "customer": "customer_master.csv",
    "inventory": "inventory_snapshot.csv",
    "promotions": "promotions.csv",
    "flags": "sku_inventory_flags.csv",
}

paths = {k: locate_file(v) for k, v in FILES.items()}
for k, p in paths.items():
    print(f"{k:12s} -> {p}")

# Read the large sales file in chunks to keep memory predictable.
sales_path = paths["sales"]
if sales_path is None:
    raise FileNotFoundError("sales_transactions.csv not found.")

sales_parts = []
for chunk in pd.read_csv(sales_path, chunksize=250_000):
    sales_parts.append(chunk)
sales = pd.concat(sales_parts, ignore_index=True)

sku_master = pd.read_csv(paths["sku"])
store_master = pd.read_csv(paths["store"])
customer_master = pd.read_csv(paths["customer"])
inventory_snapshot = pd.read_csv(paths["inventory"])
promotions = pd.read_csv(paths["promotions"])

flags = None
if paths["flags"] is not None:
    flags = pd.read_csv(paths["flags"])

print("\nShapes:")
for name, df in {
    "sales": sales,
    "sku_master": sku_master,
    "store_master": store_master,
    "customer_master": customer_master,
    "inventory_snapshot": inventory_snapshot,
    "promotions": promotions,
}.items():
    print(f"{name:20s}", df.shape)

if flags is not None:
    print(f"{'sku_inventory_flags':20s}", flags.shape)


### Optional compact synthetic fallback

The main project uses the supplied synthetic data, as required by the engagement brief. If you need a standalone demo without the supplied files, the following generator creates small relational versions of the same entities. The production notebook should still use the supplied extracts.


In [ ]:
# ============================================================
# STEP 1B — OPTIONAL compact synthetic fallback generator
# ============================================================
def generate_compact_synthetic_data(
    seed=42, n_days=365, n_stores=5, n_skus=50, n_customers=500, n_promos=10
):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_days, freq="D")

    stores = pd.DataFrame({
        "store_id": [f"ST{i:02d}" for i in range(1, n_stores + 1)],
        "store_name": [f"Store {i:02d}" for i in range(1, n_stores + 1)],
        "city": rng.choice(["Delhi", "Mumbai", "Bengaluru", "Pune"], n_stores),
        "store_type": rng.choice(["Hypermarket", "Supermarket", "Express Store"], n_stores),
        "opening_date": pd.Timestamp("2020-01-01")
    })

    categories = ["Home & Kitchen", "Dairy & Bakery", "Personal Care", "Electronics", "Toys"]
    skus = pd.DataFrame({
        "sku_id": [f"SKU{i:05d}" for i in range(1, n_skus + 1)],
        "sku_name": [f"Product {i:05d}" for i in range(1, n_skus + 1)],
        "category": rng.choice(categories, n_skus),
        "subcategory": rng.choice(["A", "B", "C"], n_skus),
        "unit_price": np.round(rng.uniform(50, 2000, n_skus), 2),
        "cost_price": np.round(rng.uniform(25, 1500, n_skus), 2),
        "brand": rng.choice(["BrandA", "BrandB", "BrandC"], n_skus)
    })
    skus["cost_price"] = np.minimum(skus["cost_price"], skus["unit_price"] * 0.8)

    customers = pd.DataFrame({
        "cust_id": [f"CUST{i:05d}" for i in range(1, n_customers + 1)],
        "age": rng.integers(18, 70, n_customers),
        "gender": rng.choice(["Male", "Female", "Other"], n_customers),
        "city": rng.choice(["Delhi", "Mumbai", "Bengaluru", "Pune"], n_customers),
        "loyalty_segment": rng.choice(["Bronze", "Silver", "Gold", "Platinum"], n_customers),
        "preferred_channel": rng.choice(["In-Store", "Online", "Mobile App"], n_customers),
        "registration_date": rng.choice(dates, n_customers)
    })

    promo_dates = rng.choice(dates[:-14], n_promos, replace=False)
    promos = pd.DataFrame({
        "promo_id": [f"PROMO{i:03d}" for i in range(1, n_promos + 1)],
        "promo_name": [f"Campaign {i:03d}" for i in range(1, n_promos + 1)],
        "start_date": promo_dates,
        "end_date": promo_dates + pd.to_timedelta(rng.integers(3, 14, n_promos), unit="D"),
        "discount_pct": rng.uniform(5, 30, n_promos).round(1),
        "promo_type": rng.choice(["Percentage Discount", "BOGO", "Bundle Offer"], n_promos),
        "target_type": "All",
        "target_value": "All"
    })

    # Generate a compact transaction table.
    n_tx = max(10_000, n_days * n_stores * 5)
    tx_dates = rng.choice(dates, n_tx)
    tx_skus = rng.choice(skus.sku_id, n_tx)
    tx_stores = rng.choice(stores.store_id, n_tx)
    tx_customers = rng.choice(customers.cust_id, n_tx)
    qty = rng.poisson(2, n_tx).clip(1, 10)
    prices = skus.set_index("sku_id").loc[tx_skus, "unit_price"].to_numpy()
    promo_mask = rng.random(n_tx) < 0.20
    discounts = np.where(promo_mask, rng.uniform(5, 25, n_tx), 0)
    sales_demo = pd.DataFrame({
        "date": tx_dates,
        "receipt_id": [f"R{i:08d}" for i in range(n_tx)],
        "store_id": tx_stores,
        "sku_id": tx_skus,
        "customer_id": tx_customers,
        "quantity": qty,
        "unit_price": prices,
        "discount_pct": discounts.round(1),
        "channel": rng.choice(["In-Store", "Online", "Mobile App"], n_tx),
        "promo_id": np.where(promo_mask, rng.choice(promos.promo_id, n_tx), None)
    })
    sales_demo["total_value"] = (
        sales_demo["quantity"] * sales_demo["unit_price"] *
        (1 - sales_demo["discount_pct"] / 100)
    )

    inv = pd.MultiIndex.from_product(
        [stores.store_id, skus.sku_id], names=["store_id", "sku_id"]
    ).to_frame(index=False)
    inv["stock_on_hand"] = rng.integers(0, 500, len(inv))
    inv["reorder_point"] = rng.integers(20, 100, len(inv))
    inv["safety_stock"] = rng.integers(5, 40, len(inv))
    inv["last_restock_date"] = dates[-1]

    return sales_demo, skus, stores, customers, inv, promos

# Example only:
# demo_sales, demo_sku, demo_store, demo_customer, demo_inventory, demo_promotions = generate_compact_synthetic_data()


In [ ]:
# ============================================================
# STEP 1C — Data cleaning and validation
# ============================================================
sales["date"] = pd.to_datetime(sales["date"], dayfirst=True, errors="coerce")
for df, col in [
    (sku_master, "sku_id"),
    (store_master, "store_id"),
    (customer_master, "cust_id"),
    (inventory_snapshot, "sku_id"),
    (inventory_snapshot, "store_id"),
    (promotions, "promo_id"),
]:
    df[col] = df[col].astype(str)

numeric_sales = ["quantity", "unit_price", "total_value", "discount_pct"]
for col in numeric_sales:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

sales = sales.dropna(subset=["date", "sku_id", "store_id", "quantity"])
sales = sales.drop_duplicates()

sku_master = sku_master.drop_duplicates("sku_id")
store_master = store_master.drop_duplicates("store_id")
customer_master = customer_master.drop_duplicates("cust_id")
inventory_snapshot = inventory_snapshot.drop_duplicates(["store_id", "sku_id"])

promotions["start_date"] = pd.to_datetime(promotions["start_date"], errors="coerce")
promotions["end_date"] = pd.to_datetime(promotions["end_date"], errors="coerce")

print("Missing values — sales:")
display(sales.isna().mean().sort_values(ascending=False).head(10))
print("Duplicate transaction rows:", sales.duplicated().sum())
print("Sales date range:", sales["date"].min().date(), "to", sales["date"].max().date())


## STEP 2 — Data manipulation, joins and feature engineering

The supplied transaction schema contains date, store, SKU, customer, quantity, price, value, channel, discount and promotion identifiers. SKU and inventory schemas provide product economics and stock-position fields. fileciteturn0file1L50-L59 fileciteturn0file1L83-L105

For forecasting, the notebook aggregates transaction demand to **weekly SKU level**. This is consistent with the engagement brief's required weekly SKU-level forecast. fileciteturn0file0L183-L186


In [ ]:
# ============================================================
# STEP 2A — Join dimensions to transactions
# ============================================================
sales_enriched = (
    sales
    .merge(
        sku_master[["sku_id", "sku_name", "category", "subcategory", "cost_price", "brand"]],
        on="sku_id", how="left", validate="many_to_one"
    )
    .merge(
        store_master[["store_id", "store_name", "city", "store_type"]],
        on="store_id", how="left", validate="many_to_one"
    )
    .merge(
        customer_master[["cust_id", "age", "gender", "loyalty_segment", "preferred_channel"]],
        left_on="customer_id", right_on="cust_id", how="left", validate="many_to_one"
    )
)

# Promotion information can be joined by promo_id.
promo_cols = ["promo_id", "promo_name", "promo_type", "target_type", "target_value"]
sales_enriched = sales_enriched.merge(
    promotions[promo_cols], on="promo_id", how="left"
)

sales_enriched["promo_flag"] = sales_enriched["promo_id"].notna().astype(int)
sales_enriched["gross_margin"] = (
    sales_enriched["total_value"] -
    sales_enriched["quantity"] * sales_enriched["cost_price"]
)

print("Analytical transaction view:", sales_enriched.shape)
display(sales_enriched.head())


In [ ]:
# ============================================================
# STEP 2B — Weekly demand panel
# ============================================================
weekly = (
    sales_enriched
    .assign(week=lambda d: d["date"].dt.to_period("W-SUN").dt.start_time)
    .groupby(["week", "sku_id"], as_index=False)
    .agg(
        demand=("quantity", "sum"),
        revenue=("total_value", "sum"),
        avg_unit_price=("unit_price", "mean"),
        avg_discount=("discount_pct", "mean"),
        promo_rate=("promo_flag", "mean"),
        active_stores=("store_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        gross_margin=("gross_margin", "sum")
    )
)

# Reindex to a complete weekly grid for each SKU so zero-sales weeks are represented.
all_weeks = pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="7D")
all_skus = sku_master["sku_id"].dropna().unique()

grid = pd.MultiIndex.from_product(
    [all_weeks, all_skus], names=["week", "sku_id"]
).to_frame(index=False)

weekly = grid.merge(weekly, on=["week", "sku_id"], how="left")

for col in ["demand", "revenue", "avg_discount", "promo_rate", "gross_margin"]:
    weekly[col] = weekly[col].fillna(0)

for col in ["avg_unit_price", "active_stores", "unique_customers"]:
    weekly[col] = weekly[col].fillna(0)

weekly = weekly.merge(
    sku_master[["sku_id", "category", "subcategory", "cost_price", "unit_price", "brand"]],
    on="sku_id", how="left"
)

weekly["week_of_year"] = weekly["week"].dt.isocalendar().week.astype(int)
weekly["month"] = weekly["week"].dt.month
weekly["quarter"] = weekly["week"].dt.quarter
weekly["year"] = weekly["week"].dt.year

# Cyclic seasonality features.
weekly["week_sin"] = np.sin(2 * np.pi * weekly["week_of_year"] / 52)
weekly["week_cos"] = np.cos(2 * np.pi * weekly["week_of_year"] / 52)

# Sort before all lag/rolling calculations to avoid leakage.
weekly = weekly.sort_values(["sku_id", "week"]).reset_index(drop=True)

for lag in [1, 2, 4, 8, 12, 52]:
    weekly[f"demand_lag_{lag}"] = weekly.groupby("sku_id")["demand"].shift(lag)

for window in [4, 8, 12]:
    weekly[f"demand_roll_mean_{window}"] = (
        weekly.groupby("sku_id")["demand"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=2).mean())
    )
    weekly[f"demand_roll_std_{window}"] = (
        weekly.groupby("sku_id")["demand"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=2).std())
    )

weekly["demand_growth_4w"] = (
    weekly["demand_lag_1"] / weekly["demand_lag_4"].replace(0, np.nan) - 1
).replace([np.inf, -np.inf], np.nan)

print("Weekly panel shape:", weekly.shape)
display(weekly.head(10))


In [ ]:
# ============================================================
# STEP 2C — Latest inventory and inventory feature engineering
# ============================================================
inventory = inventory_snapshot.copy()
inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"], errors="coerce"
)

# Current snapshot: if duplicate snapshots exist, keep the most recent restock record.
inventory_latest = (
    inventory.sort_values("last_restock_date")
    .groupby(["store_id", "sku_id"], as_index=False)
    .tail(1)
)

sku_inventory = (
    inventory_latest
    .groupby("sku_id", as_index=False)
    .agg(
        stock_on_hand=("stock_on_hand", "sum"),
        reorder_point=("reorder_point", "sum"),
        safety_stock=("safety_stock", "sum"),
        stores_with_inventory=("store_id", "nunique")
    )
)

weekly = weekly.merge(sku_inventory, on="sku_id", how="left")
weekly[["stock_on_hand", "reorder_point", "safety_stock"]] = (
    weekly[["stock_on_hand", "reorder_point", "safety_stock"]].fillna(0)
)

print("Final analytical weekly view:", weekly.shape)


## STEP 3 — Exploratory Data Analysis

The client brief asks for demand patterns, seasonality, top movers, dead stock and business-relevant insights, with readable charts. fileciteturn0file0L297-L302


In [ ]:
# ============================================================
# STEP 3A — Weekly total demand trend
# ============================================================
weekly_total = weekly.groupby("week", as_index=False)["demand"].sum()

plt.figure(figsize=(14, 5))
sns.lineplot(data=weekly_total, x="week", y="demand", linewidth=2)
plt.title("FORESIGHT — Weekly Total Demand Trend")
plt.xlabel("Week")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 3B — Demand distribution by category
# ============================================================
category_demand = (
    weekly.groupby(["category", "sku_id"], as_index=False)["demand"].sum()
)

plt.figure(figsize=(13, 6))
sns.boxplot(data=category_demand, x="category", y="demand")
plt.xticks(rotation=35, ha="right")
plt.title("Demand Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Total Units Sold per SKU")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 3C — Promotion impact boxplot
# ============================================================
promo_weekly = weekly.copy()
promo_weekly["promotion_status"] = np.where(
    promo_weekly["promo_rate"] > 0, "Promotion", "No Promotion"
)

# Compare non-zero demand to avoid a zero-inflated visual dominating the plot.
promo_plot = promo_weekly[promo_weekly["demand"] >= 0].copy()

plt.figure(figsize=(9, 6))
sns.boxplot(data=promo_plot, x="promotion_status", y="demand")
plt.title("Promotion Impact on Weekly Demand")
plt.xlabel("")
plt.ylabel("Weekly Units Sold")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 3D — Unit cost vs demand scatterplot
# ============================================================
sku_summary = (
    weekly.groupby("sku_id", as_index=False)
    .agg(
        total_demand=("demand", "sum"),
        unit_cost=("cost_price", "first"),
        category=("category", "first")
    )
)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=sku_summary,
    x="unit_cost",
    y="total_demand",
    hue="category",
    alpha=0.65,
    s=55
)
plt.title("Unit Cost vs Total Demand by SKU")
plt.xlabel("Unit Cost")
plt.ylabel("Total Units Sold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 3E — Business insight tables
# ============================================================
top_movers = (
    sku_summary.sort_values("total_demand", ascending=False)
    .head(20)
    .merge(sku_master[["sku_id", "sku_name", "category"]], on=["sku_id", "category"], how="left")
)

dead_stock = (
    sku_summary.sort_values("total_demand", ascending=True)
    .head(20)
    .merge(sku_inventory, on="sku_id", how="left")
    .merge(sku_master[["sku_id", "sku_name", "category"]], on="sku_id", how="left")
)

print("Top movers")
display(top_movers[["sku_id", "sku_name", "category", "total_demand"]].head(10))

print("Potential slow movers / dead-stock candidates")
display(dead_stock[["sku_id", "sku_name", "category", "total_demand", "stock_on_hand"]].head(10))


## STEP 4 — Forecasting: seasonal-naive baseline + LightGBM + LSTM

The engagement brief requires a seasonal-naive baseline, rolling-origin evaluation, WAPE, and strict leakage prevention. fileciteturn0file0L236-L254

This notebook includes:
- **Seasonal-naive baseline:** same week last year (`lag_52`).
- **LightGBM:** scalable tabular benchmark using lag/rolling/calendar/promotion features.
- **LSTM:** deep-learning demonstration using 12-week sequential histories for the highest-volume SKUs.

Because the supplied transaction extract is large, the LSTM is deliberately limited to the top-volume SKUs for practical notebook runtime. The replenishment engine still produces an action for every SKU using LSTM forecasts where available and a transparent rolling-demand fallback for other SKUs.


In [ ]:
# ============================================================
# STEP 4A — Forecast metric helpers
# ============================================================
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true).sum()
    return np.abs(y_true - y_pred).sum() / denom if denom else np.nan

def bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(y_pred - y_true)

# Final holdout: last 8 weeks.
FORECAST_HORIZON_WEEKS = 8
cutoff = weekly["week"].max() - pd.Timedelta(weeks=FORECAST_HORIZON_WEEKS - 1)

train_df = weekly[weekly["week"] < cutoff].copy()
test_df = weekly[weekly["week"] >= cutoff].copy()

test_df["seasonal_naive"] = test_df["demand_lag_52"]

baseline_valid = test_df.dropna(subset=["seasonal_naive"])
baseline_wape = wape(baseline_valid["demand"], baseline_valid["seasonal_naive"])
baseline_bias = bias(baseline_valid["demand"], baseline_valid["seasonal_naive"])

print("Baseline WAPE:", round(baseline_wape, 4))
print("Baseline Bias:", round(baseline_bias, 4))


In [ ]:
# ============================================================
# STEP 4B — LightGBM benchmark
# ============================================================
from lightgbm import LGBMRegressor

feature_cols = [
    "demand_lag_1", "demand_lag_2", "demand_lag_4",
    "demand_lag_8", "demand_lag_12", "demand_lag_52",
    "demand_roll_mean_4", "demand_roll_std_4",
    "demand_roll_mean_8", "demand_roll_std_8",
    "demand_roll_mean_12", "demand_roll_std_12",
    "demand_growth_4w", "promo_rate", "avg_discount",
    "week_of_year", "month", "quarter", "week_sin", "week_cos",
    "cost_price", "unit_price"
]

model_df = weekly.dropna(subset=feature_cols).copy()

# Chronological split — never random shuffle time-series rows.
lgb_train = model_df[model_df["week"] < cutoff]
lgb_test = model_df[model_df["week"] >= cutoff]

X_train = lgb_train[feature_cols]
y_train = lgb_train["demand"]
X_test = lgb_test[feature_cols]
y_test = lgb_test["demand"]

lgb_model = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)

lgb_model.fit(X_train, y_train)
lgb_test["lgb_forecast"] = np.clip(lgb_model.predict(X_test), 0, None)

lgb_wape = wape(lgb_test["demand"], lgb_test["lgb_forecast"])
lgb_bias = bias(lgb_test["demand"], lgb_test["lgb_forecast"])

comparison = pd.DataFrame({
    "Model": ["Seasonal Naive", "LightGBM"],
    "WAPE": [baseline_wape, lgb_wape],
    "Bias": [baseline_bias, lgb_bias]
})
display(comparison)


In [ ]:
# ============================================================
# STEP 4C — LSTM preparation
# ============================================================
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from sklearn.preprocessing import MinMaxScaler

    tf.random.set_seed(42)

    # Top SKUs by historical demand.
    TOP_N_LSTM_SKUS = 50
    top_lstm_skus = (
        weekly.groupby("sku_id")["demand"]
        .sum()
        .nlargest(TOP_N_LSTM_SKUS)
        .index
        .tolist()
    )

    lstm_panel = weekly[weekly["sku_id"].isin(top_lstm_skus)].copy()
    lstm_panel = lstm_panel.sort_values(["sku_id", "week"])

    SEQ_LEN = 12
    LSTM_FEATURES = ["demand", "promo_rate", "avg_discount"]

    X_seq, y_seq, meta = [], [], []

    for sku_id, g in lstm_panel.groupby("sku_id"):
        g = g.sort_values("week").reset_index(drop=True)
        vals = g[LSTM_FEATURES].fillna(0).astype(float).values

        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(vals)

        # Store per-SKU scaler for inverse transform of the target.
        for i in range(SEQ_LEN, len(g)):
            X_seq.append(scaled[i-SEQ_LEN:i])
            y_seq.append(scaled[i, 0])
            meta.append((sku_id, g.loc[i, "week"], scaler))

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=np.float32)

    meta_df = pd.DataFrame(meta, columns=["sku_id", "week", "scaler"])

    # Chronological split by week.
    train_mask = meta_df["week"] < cutoff
    test_mask = meta_df["week"] >= cutoff

    X_lstm_train = X_seq[train_mask.to_numpy()]
    y_lstm_train = y_seq[train_mask.to_numpy()]
    X_lstm_test = X_seq[test_mask.to_numpy()]
    y_lstm_test = y_seq[test_mask.to_numpy()]
    meta_test = meta_df.loc[test_mask].reset_index(drop=True)

    print("LSTM train shape:", X_lstm_train.shape)
    print("LSTM test shape :", X_lstm_test.shape)

except ImportError:
    print("TensorFlow is unavailable in this environment.")
    print("Install it with: pip install tensorflow")
    print("Then rerun this cell and the following LSTM cells.")


In [ ]:
# ============================================================
# STEP 4D — LSTM model training
# ============================================================
try:
    # A compact architecture for a reproducible notebook.
    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(LSTM_FEATURES))),
        Dropout(0.20),
        LSTM(32),
        Dropout(0.20),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    lstm_model.compile(
        optimizer="adam",
        loss="mae"
    )

    history = lstm_model.fit(
        X_lstm_train,
        y_lstm_train,
        validation_split=0.10,
        epochs=15,
        batch_size=256,
        verbose=1,
        shuffle=False
    )

    pred_scaled = lstm_model.predict(X_lstm_test, verbose=0).ravel()

    # Invert each prediction using the corresponding SKU scaler.
    pred_actual = []
    actual_actual = []

    for i, row in meta_test.iterrows():
        scaler = row["scaler"]
        p = scaler.inverse_transform(
            np.column_stack([
                [pred_scaled[i]],
                [0.0],
                [0.0]
            ])
        )[0, 0]
        a = scaler.inverse_transform(
            np.column_stack([
                [y_lstm_test[i]],
                [0.0],
                [0.0]
            ])
        )[0, 0]
        pred_actual.append(max(0, p))
        actual_actual.append(max(0, a))

    lstm_results = meta_test[["sku_id", "week"]].copy()
    lstm_results["actual"] = actual_actual
    lstm_results["lstm_forecast"] = pred_actual

    lstm_wape = wape(lstm_results["actual"], lstm_results["lstm_forecast"])
    lstm_bias = bias(lstm_results["actual"], lstm_results["lstm_forecast"])

    print("LSTM WAPE:", round(lstm_wape, 4))
    print("LSTM Bias:", round(lstm_bias, 4))

except NameError:
    print("Run the LSTM preparation cell after installing TensorFlow.")


In [ ]:
# ============================================================
# STEP 4E — Forecast vs actual visualization
# ============================================================
try:
    example_sku = (
        lstm_results.groupby("sku_id")["actual"].sum()
        .sort_values(ascending=False)
        .index[0]
    )
    plot_df = lstm_results[lstm_results["sku_id"] == example_sku].sort_values("week")

    plt.figure(figsize=(13, 5))
    plt.plot(plot_df["week"], plot_df["actual"], marker="o", label="Actual")
    plt.plot(plot_df["week"], plot_df["lstm_forecast"], marker="o", label="LSTM Forecast")
    plt.title(f"LSTM Forecast vs Actual — {example_sku}")
    plt.xlabel("Week")
    plt.ylabel("Units")
    plt.legend()
    plt.tight_layout()
    plt.show()
except NameError:
    print("LSTM results are not available yet.")


### Backtesting note

The brief's non-negotiable rule is to compare the model against the seasonal-naive baseline using time-aware backtesting and to ship the baseline instead if the complex model does not win. fileciteturn0file0L250-L254

The code below reports the holdout WAPE and makes the selection explicit rather than hiding a weak model.


In [ ]:
# ============================================================
# STEP 4F — Model selection summary
# ============================================================
rows = [{"Model": "Seasonal Naive", "WAPE": baseline_wape, "Bias": baseline_bias}]
if "lgb_wape" in globals():
    rows.append({"Model": "LightGBM", "WAPE": lgb_wape, "Bias": lgb_bias})
if "lstm_wape" in globals():
    rows.append({"Model": "LSTM", "WAPE": lstm_wape, "Bias": lstm_bias})

model_scores = pd.DataFrame(rows).sort_values("WAPE")
display(model_scores)

winner = model_scores.iloc[0]["Model"]
print(f"Selected model under the WAPE rule: {winner}")


## STEP 5 — Inventory Optimization: ROP + EOQ + automated actions

The supplied inventory snapshot contains on-hand units, reorder point, safety stock and last-restock date. It does **not** contain lead time or annual holding/order costs, so this notebook makes those assumptions explicit instead of pretending they came from the source. fileciteturn0file1L83-L90

The project brief's risk logic is to compare forecast demand over lead time with on-hand plus on-order inventory, and to attach a recommended action and rupee value at stake. fileciteturn0file0L260-L284

For this implementation:
- Default lead time = 7 days (configurable).
- Service level = 95%.
- Annual holding cost = 20% of SKU cost.
- Ordering cost = ₹500 per replenishment.
- Safety stock uses demand variability when enough history exists.
- No purchase order is placed automatically; the dashboard only recommends an action, consistent with the project scope. fileciteturn0file0L187-L192


In [ ]:
# ============================================================
# STEP 5A — Inventory optimization parameters
# ============================================================
DEFAULT_LEAD_TIME_DAYS = 7
SERVICE_LEVEL = 0.95
Z_VALUE = 1.645  # approx. 95% one-sided service level
ANNUAL_HOLDING_RATE = 0.20
ORDERING_COST = 500.0
FORWARD_WEEKS = 8

# Aggregate inventory to SKU level.
current_inventory = (
    inventory_snapshot
    .groupby("sku_id", as_index=False)
    .agg(
        stock_on_hand=("stock_on_hand", "sum"),
        reorder_point_source=("reorder_point", "sum"),
        safety_stock_source=("safety_stock", "sum")
    )
)

sku_demand_stats = (
    weekly.groupby("sku_id", as_index=False)
    .agg(
        avg_weekly_demand=("demand", "mean"),
        std_weekly_demand=("demand", "std"),
        last_12w_demand=("demand", lambda s: s.tail(12).mean())
    )
)

sku_opt = (
    current_inventory
    .merge(sku_demand_stats, on="sku_id", how="left")
    .merge(
        sku_master[["sku_id", "sku_name", "category", "cost_price", "unit_price"]],
        on="sku_id", how="left"
    )
)

sku_opt["avg_weekly_demand"] = sku_opt["avg_weekly_demand"].fillna(0)
sku_opt["std_weekly_demand"] = sku_opt["std_weekly_demand"].fillna(0)
sku_opt["last_12w_demand"] = sku_opt["last_12w_demand"].fillna(0)
sku_opt["daily_demand"] = sku_opt["last_12w_demand"] / 7

sku_opt["lead_time_demand"] = sku_opt["daily_demand"] * DEFAULT_LEAD_TIME_DAYS
sku_opt["daily_std"] = sku_opt["std_weekly_demand"] / np.sqrt(7)

sku_opt["calculated_safety_stock"] = (
    Z_VALUE * sku_opt["daily_std"] * np.sqrt(DEFAULT_LEAD_TIME_DAYS)
)

sku_opt["ROP"] = (
    sku_opt["lead_time_demand"] +
    sku_opt["calculated_safety_stock"]
)

# EOQ: sqrt(2DS/H)
# D = annual demand; S = order cost; H = annual holding cost per unit.
sku_opt["annual_demand"] = sku_opt["last_12w_demand"] * 52
sku_opt["holding_cost_per_unit_year"] = (
    sku_opt["cost_price"] * ANNUAL_HOLDING_RATE
).clip(lower=0.01)

sku_opt["EOQ"] = np.sqrt(
    (2 * sku_opt["annual_demand"] * ORDERING_COST) /
    sku_opt["holding_cost_per_unit_year"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

sku_opt["forecast_8w_demand"] = sku_opt["last_12w_demand"] * FORWARD_WEEKS

sku_opt["stockout_risk"] = sku_opt["stock_on_hand"] < sku_opt["ROP"]
sku_opt["overstock_risk"] = (
    sku_opt["stock_on_hand"] >
    sku_opt["forecast_8w_demand"] + sku_opt["calculated_safety_stock"] + sku_opt["EOQ"]
)

def action_rule(row):
    if row["stockout_risk"] and not row["overstock_risk"]:
        return "REORDER NOW"
    if row["overstock_risk"] and not row["stockout_risk"]:
        return "MARKDOWN / CLEAR"
    if row["stockout_risk"] and row["overstock_risk"]:
        return "WATCH / VOLATILE"
    return "HEALTHY"

sku_opt["recommended_action"] = sku_opt.apply(action_rule, axis=1)

sku_opt["reorder_qty"] = np.where(
    sku_opt["recommended_action"].eq("REORDER NOW"),
    np.maximum(sku_opt["EOQ"], sku_opt["ROP"] + sku_opt["EOQ"] - sku_opt["stock_on_hand"]),
    0
)

sku_opt["sales_at_risk_rupees"] = (
    np.maximum(sku_opt["ROP"] - sku_opt["stock_on_hand"], 0) *
    sku_opt["unit_price"]
)

sku_opt["locked_capital_rupees"] = (
    np.maximum(sku_opt["stock_on_hand"] - sku_opt["forecast_8w_demand"], 0) *
    sku_opt["cost_price"]
)

sku_opt["priority_score"] = (
    sku_opt["sales_at_risk_rupees"] + sku_opt["locked_capital_rupees"]
)

sku_opt = sku_opt.sort_values("priority_score", ascending=False)

display(
    sku_opt[
        [
            "sku_id", "sku_name", "category", "stock_on_hand",
            "ROP", "EOQ", "forecast_8w_demand",
            "recommended_action", "reorder_qty",
            "sales_at_risk_rupees", "locked_capital_rupees"
        ]
    ].head(25)
)


In [ ]:
# ============================================================
# STEP 5B — Optional reconciliation against anomaly ground truth
# ============================================================
if flags is not None:
    truth = flags[["sku_id", "flag"]].drop_duplicates()

    eval_df = sku_opt.merge(truth, on="sku_id", how="left")
    eval_df["ground_truth"] = eval_df["flag"].fillna("HEALTHY")

    # Simple business-rule score:
    eval_df["predicted_risk"] = np.where(
        eval_df["recommended_action"].isin(["REORDER NOW", "WATCH / VOLATILE"]),
        "STOCKOUT_RISK",
        np.where(
            eval_df["recommended_action"].eq("MARKDOWN / CLEAR"),
            "SLOW_MOVER",
            "HEALTHY"
        )
    )

    display(
        pd.crosstab(
            eval_df["ground_truth"],
            eval_df["predicted_risk"],
            rownames=["Ground Truth"],
            colnames=["FORESIGHT Action Class"]
        )
    )
else:
    print("No sku_inventory_flags.csv found; skipping anomaly-ground-truth evaluation.")


In [ ]:
# ============================================================
# STEP 5C — Decisioning grid
# ============================================================
grid_df = sku_opt.copy()
grid_df["stockout_pressure"] = (
    grid_df["ROP"] - grid_df["stock_on_hand"]
)
grid_df["overstock_pressure"] = (
    grid_df["stock_on_hand"] - grid_df["forecast_8w_demand"]
)

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=grid_df,
    x="overstock_pressure",
    y="stockout_pressure",
    hue="recommended_action",
    size="priority_score",
    sizes=(30, 300),
    alpha=0.75
)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("FORESIGHT Decisioning Grid — Overstock vs Stockout Pressure")
plt.xlabel("Overstock Pressure (positive = excess stock)")
plt.ylabel("Stockout Pressure (positive = below ROP)")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 5D — Save reproducible outputs for Streamlit
# ============================================================
sku_opt.to_csv(OUTPUT_DIR / "replenishment_recommendations.csv", index=False)

# Forecast output: LightGBM holdout results if available.
if "lgb_test" in globals():
    forecast_export = lgb_test[
        ["week", "sku_id", "demand", "lgb_forecast"]
    ].rename(columns={"lgb_forecast": "forecast"})
elif "lstm_results" in globals():
    forecast_export = lstm_results.rename(
        columns={"lstm_forecast": "forecast"}
    )
else:
    forecast_export = test_df[
        ["week", "sku_id", "demand", "seasonal_naive"]
    ].rename(columns={"seasonal_naive": "forecast"})

forecast_export.to_csv(OUTPUT_DIR / "forecast_results.csv", index=False)
model_scores.to_csv(OUTPUT_DIR / "model_scores.csv", index=False)

print("Saved:")
for p in [
    OUTPUT_DIR / "replenishment_recommendations.csv",
    OUTPUT_DIR / "forecast_results.csv",
    OUTPUT_DIR / "model_scores.csv"
]:
    print(" -", p)


# STEP 6 — Streamlit Dashboard Deployment

The engagement brief requires a filterable planning dashboard with forecast vs actual, risk flags and a prioritised reorder/markdown list. fileciteturn0file0L315-L320

The dashboard below is intentionally decision-oriented:
- KPI cards for reorder count, estimated sales at risk and locked capital.
- Category/SKU filters.
- Reorder / clearance table.
- Forecast vs actual chart.
- CSV download for the replenishment report.

It **does not place purchase orders**, because automated purchase-order placement is explicitly out of scope. fileciteturn0file0L187-L192


In [ ]:
# ============================================================
# STEP 6A — Streamlit app source preview
# ============================================================
STREAMLIT_APP = r'''
import streamlit as st
import pandas as pd
import plotly.express as px
from pathlib import Path

st.set_page_config(
    page_title="FORESIGHT — Demand & Inventory Intelligence",
    page_icon="📦",
    layout="wide"
)

BASE = Path(__file__).resolve().parents[1]
OUT = BASE / "outputs"

repl = pd.read_csv(OUT / "replenishment_recommendations.csv")
forecast = pd.read_csv(OUT / "forecast_results.csv")
forecast["week"] = pd.to_datetime(forecast["week"])

st.title("FORESIGHT")
st.caption("AI-Powered Demand & Inventory Intelligence Platform")

# Sidebar filters
st.sidebar.header("Filters")
categories = ["All"] + sorted(repl["category"].dropna().unique().tolist())
selected_category = st.sidebar.selectbox("Category", categories)

if selected_category != "All":
    repl_view = repl[repl["category"] == selected_category].copy()
else:
    repl_view = repl.copy()

sku_options = ["All"] + sorted(repl_view["sku_id"].unique().tolist())
selected_sku = st.sidebar.selectbox("SKU", sku_options)

if selected_sku != "All":
    repl_view = repl_view[repl_view["sku_id"] == selected_sku].copy()

# KPI cards
reorder_count = int((repl_view["recommended_action"] == "REORDER NOW").sum())
sales_risk = repl_view["sales_at_risk_rupees"].sum()
locked_capital = repl_view["locked_capital_rupees"].sum()

c1, c2, c3 = st.columns(3)
c1.metric("SKUs to Reorder", f"{reorder_count:,}")
c2.metric("Sales at Risk", f"₹{sales_risk:,.0f}")
c3.metric("Locked Capital", f"₹{locked_capital:,.0f}")

st.subheader("Prioritised Replenishment / Clearance")
display_cols = [
    "sku_id", "sku_name", "category", "stock_on_hand",
    "ROP", "EOQ", "forecast_8w_demand",
    "recommended_action", "reorder_qty",
    "sales_at_risk_rupees", "locked_capital_rupees"
]
st.dataframe(
    repl_view[display_cols].sort_values(
        "sales_at_risk_rupees", ascending=False
    ),
    use_container_width=True,
    hide_index=True
)

if selected_sku != "All":
    f = forecast[forecast["sku_id"] == selected_sku].sort_values("week")
else:
    f = forecast.groupby("week", as_index=False).agg(
        demand=("demand", "sum"),
        forecast=("forecast", "sum")
    )

if not f.empty:
    st.subheader("Forecast vs Actual")
    long_df = f.rename(columns={"demand": "Actual", "forecast": "Forecast"})
    fig = px.line(
        long_df,
        x="week",
        y=["Actual", "Forecast"],
        markers=True,
        title="Demand Forecast vs Actual"
    )
    st.plotly_chart(fig, use_container_width=True)

st.subheader("Download Replenishment Report")
csv_bytes = repl_view.to_csv(index=False).encode("utf-8")
st.download_button(
    "Download CSV Report",
    data=csv_bytes,
    file_name="foresight_replenishment_report.csv",
    mime="text/csv"
)

st.info(
    "FORESIGHT provides recommendations for human review. "
    "It does not place purchase orders automatically."
)
'''

print(STREAMLIT_APP[:3000])


## Deployment commands

From the project root:

```bash
pip install -r requirements.txt
jupyter notebook
```

After the notebook has generated the files in `outputs/`:

```bash
streamlit run app/streamlit_app.py
```

For deployment, the engagement brief lists Streamlit Community Cloud, Hugging Face Spaces and Render as suitable options. fileciteturn0file0L471-L479

### Suggested repository structure

```text
foresight/
├── data/
│   ├── sales_transactions.csv
│   ├── sku_master.csv
│   ├── store_master.csv
│   ├── customer_master.csv
│   ├── inventory_snapshot.csv
│   ├── promotions.csv
│   └── sku_inventory_flags.csv
├── notebooks/
│   └── FORESIGHT_End_to_End.ipynb
├── outputs/
│   ├── forecast_results.csv
│   ├── model_scores.csv
│   └── replenishment_recommendations.csv
├── app/
│   └── streamlit_app.py
├── README.md
└── requirements.txt
```

This structure follows the engagement brief's suggested separation of pipeline, forecasting, risk, dashboard, service and reports. fileciteturn0file0L483-L495


## Final acceptance checklist

- [x] Data ingestion and cleaning are coded and reproducible.
- [x] Relational joins across sales, SKU, store, customer, promotion and inventory data.
- [x] Weekly SKU-level demand panel.
- [x] Lag and rolling features with leakage-safe `shift(1)`.
- [x] Professional EDA visualisations.
- [x] Seasonal-naive baseline.
- [x] LightGBM benchmark.
- [x] LSTM demand forecasting path.
- [x] WAPE and bias evaluation.
- [x] ROP and EOQ logic.
- [x] Stockout / overstock action classification.
- [x] Rupee impact estimates.
- [x] Streamlit dashboard source.
- [x] CSV replenishment report generation.
- [x] No automatic purchase-order placement.

The client acceptance criteria explicitly require a weekly SKU-level forecast, baseline comparison, rolling-origin backtesting, leakage prevention, transparent risk logic, and a usable dashboard. fileciteturn0file0L303-L320
